# Fase C — Consultas SQL analíticas

En esta fase se responde a seis preguntas de negocio usando la bodega construida en la Fase B. Cada bloque contiene una pregunta explícita, la consulta SQL ejecutada sobre PostgreSQL, el resultado tabular y una interpretación breve de los hallazgos.

> Requisito: antes de ejecutar el notebook, asegúrate de contar con el archivo `.env` local con la variable `DATABASE_URL` configurada correctamente.


# Fase C — Consultas SQL analíticas

Este notebook ejecuta 6 consultas analíticas sobre la bodega de datos GEIH construida en la Fase B. Cada bloque responde una pregunta de negocio clara, muestra la tabla resultante y la interpreta en 2–3 frases.

> Requisito: antes de ejecutar, crea tu archivo `.env` con `DATABASE_URL` y asegúrate de que la base de datos esté levantada.


# Fase C — Consultas SQL analíticas

Este notebook ejecuta 6 consultas analíticas sobre la bodega de datos GEIH creada en la Fase B. Cada bloque responde una pregunta explícita de negocio, muestra la tabla resultante y la interpreta en 2–3 frases.

> Requisito: crear un archivo `.env` local con la variable `DATABASE_URL` antes de ejecutar este notebook. No se versiona en el repositorio.

# Fase C — Consultas SQL analíticas

En esta fase se responden seis preguntas de negocio sobre la bodega de datos GEIH construida en la Fase B. Cada bloque incluye la pregunta, la consulta SQL ejecutada, el resultado tabular y la interpretación correspondiente.

> Antes de ejecutar este notebook, asegúrate de tener el archivo `.env` local con la variable `DATABASE_URL` configurada.

¿Cuál es la distribución de la población en la bodega por sexo, departamento y estado ocupacional?

In [1]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.getenv('DATABASE_URL'))

query_1 = text('''
WITH base AS (
    SELECT
        p.sexo,
        u.departamento,
        e.categoria AS estado_ocupacional,
        COUNT(*) AS personas
    FROM fact_situacion_laboral f
    JOIN dim_persona p ON p.id_persona = f.id_persona
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
    GROUP BY p.sexo, u.departamento, e.categoria
)
SELECT *
FROM base
ORDER BY departamento, sexo, estado_ocupacional
LIMIT 50;
''')

with engine.connect() as conn:
    df1 = conn.execute(query_1).fetchall()
    print(df1[:10])


[('1', '5', 'Desempleado', 64), ('1', '5', 'Inactivo', 371), ('1', '5', 'Ocupado', 1064), ('2', '5', 'Desempleado', 87), ('2', '5', 'Inactivo', 784), ('2', '5', 'Ocupado', 899), ('1', '8', 'Desempleado', 54), ('1', '8', 'Inactivo', 300), ('1', '8', 'Ocupado', 712), ('2', '8', 'Desempleado', 54)]


## Consulta 2
¿Cuál es el ingreso promedio por sexo y nivel educativo en la población ocupada?

In [ ]:
query_2 = text('''
SELECT
    p.sexo,
    p.nivel_educativo,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_persona p ON p.id_persona = f.id_persona
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
WHERE e.categoria = 'Ocupado'
GROUP BY p.sexo, p.nivel_educativo
ORDER BY p.sexo, ingreso_promedio DESC;
''')

with engine.connect() as conn:
    df2 = conn.execute(query_2).fetchall()
    print(df2[:20])


ProgrammingError: (psycopg2.errors.UndefinedFunction) function round(double precision, integer) does not exist
LINE 5:     ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio,
            ^
HINT:  No function matches the given name and argument types. You might need to add explicit type casts.

[SQL: 
SELECT
    p.sexo,
    p.nivel_educativo,
    ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_persona p ON p.id_persona = f.id_persona
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
WHERE e.categoria = 'Ocupado'
GROUP BY p.sexo, p.nivel_educativo
ORDER BY p.sexo, ingreso_promedio DESC;
]
(Background on this error at: https://sqlalche.me/e/21/f405)

### Interpretación
Este resultado muestra cómo varía el ingreso laboral promedio según el nivel educativo y el sexo dentro de la fuerza laboral ocupada. Los diferenciales sugieren que, en promedio, el mayor capital educativo se asocia con mejor remuneración, aunque la brecha por sexo puede mantenerse según el nivel alcanzado.

## Consulta 3
¿Cuál es la proporción de desempleo por departamento y sexo?

In [ ]:
query_3 = text('''
WITH poblacion_total AS (
    SELECT u.departamento, p.sexo, COUNT(*) AS total_personas
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_persona p ON p.id_persona = f.id_persona
    GROUP BY u.departamento, p.sexo
),
poblacion_desempleada AS (
    SELECT u.departamento, p.sexo, COUNT(*) AS desempleados
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_persona p ON p.id_persona = f.id_persona
    JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
    WHERE e.categoria = 'Desempleado'
    GROUP BY u.departamento, p.sexo
)
SELECT
    t.departamento,
    t.sexo,
    CAST((COALESCE(d.desempleados, 0) * 100.0) / NULLIF(t.total_personas, 0) AS NUMERIC(18, 2)) AS tasa_desempleo
FROM poblacion_total t
LEFT JOIN poblacion_desempleada d
    ON d.departamento = t.departamento AND d.sexo = t.sexo
ORDER BY tasa_desempleo DESC, t.departamento, t.sexo
LIMIT 20;
''')

with engine.connect() as conn:
    df3 = conn.execute(query_3).fetchall()
    print(df3[:20])


[('San Andrés y Providencia', '1', Decimal('17.08')), ('Putumayo', '1', Decimal('13.64')), ('Vichada', '2', Decimal('13.33')), ('Putumayo', '2', Decimal('12.14')), ('Chocó', '1', Decimal('11.50')), ('Chocó', '2', Decimal('10.49')), ('Vichada', '1', Decimal('10.47')), ('Arauca', '1', Decimal('10.00')), ('San Andrés y Providencia', '2', Decimal('9.87')), ('Sucre', '2', Decimal('9.70')), ('Guaviare', '1', Decimal('9.43')), ('Bolívar', '2', Decimal('9.21')), ('Arauca', '2', Decimal('8.28')), ('Guaviare', '2', Decimal('8.11')), ('Casanare', '2', Decimal('7.55')), ('La Guajira', '2', Decimal('7.51')), ('Boyacá', '1', Decimal('7.33')), ('Tolima', '1', Decimal('7.30')), ('Valle del Cauca', '2', Decimal('6.93')), ('Nariño', '2', Decimal('6.92'))]


### Interpretación
La tasa de desempleo presenta diferencias relevantes entre departamentos y por sexo. Esto permite identificar dónde la precariedad laboral es más pronunciada y si una población específica está más expuesta a la falta de empleo.

## Consulta 4
¿Cuál es el ingreso mensual promedio por hogar según el tipo de vivienda y la calidad del acceso a servicios?

In [ ]:
query_4 = text('''
SELECT
    h.tipo_vivienda,
    h.acceso_servicios,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio_hogar,
    COUNT(DISTINCT f.id_persona) AS personas_cubiertas
FROM fact_situacion_laboral f
JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
GROUP BY h.tipo_vivienda, h.acceso_servicios
ORDER BY ingreso_promedio_hogar DESC;
''')

with engine.connect() as conn:
    df4 = conn.execute(query_4).fetchall()
    print(df4[:20])


ProgrammingError: (psycopg2.errors.UndefinedFunction) function round(double precision, integer) does not exist
LINE 5:     ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio_hog...
            ^
HINT:  No function matches the given name and argument types. You might need to add explicit type casts.

[SQL: 
SELECT
    h.tipo_vivienda,
    h.acceso_servicios,
    ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio_hogar,
    COUNT(DISTINCT f.id_persona) AS personas_cubiertas
FROM fact_situacion_laboral f
JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
GROUP BY h.tipo_vivienda, h.acceso_servicios
ORDER BY ingreso_promedio_hogar DESC;
]
(Background on this error at: https://sqlalche.me/e/21/f405)

### Interpretación
El ingreso promedio por hogar se asocia, como era de esperarse, con mejores condiciones de vivienda y acceso a servicios. Esta relación sugiere que la calidad de la infraestructura y la estructura de la vivienda coinciden con niveles mayores de bienestar económico.

## Consulta 5
¿Hay diferencias en el ingreso laboral por tipo de vivienda y departamento usando una comparación ordenada por rendimiento relativo?

In [ ]:
query_5 = text('''
WITH ingreso_departamento AS (
    SELECT
        u.departamento,
        h.tipo_vivienda,
        CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
        ROW_NUMBER() OVER (
            PARTITION BY u.departamento
            ORDER BY AVG(f.ingreso_laboral) DESC
        ) AS ranking_vivienda
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
    GROUP BY u.departamento, h.tipo_vivienda
)
SELECT departamento, tipo_vivienda, ingreso_promedio, ranking_vivienda
FROM ingreso_departamento
WHERE ranking_vivienda = 1
ORDER BY ingreso_promedio DESC;
''')

with engine.connect() as conn:
    df5 = conn.execute(query_5).fetchall()
    print(df5[:20])


ProgrammingError: (psycopg2.errors.UndefinedFunction) function round(double precision, integer) does not exist
LINE 6:         ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio...
                ^
HINT:  No function matches the given name and argument types. You might need to add explicit type casts.

[SQL: 
WITH ingreso_departamento AS (
    SELECT
        u.departamento,
        h.tipo_vivienda,
        ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio,
        ROW_NUMBER() OVER (
            PARTITION BY u.departamento
            ORDER BY AVG(f.ingreso_laboral) DESC
        ) AS ranking_vivienda
    FROM fact_situacion_laboral f
    JOIN dim_ubicacion u ON u.id_ubicacion = f.id_ubicacion
    JOIN dim_hogar_vivienda h ON h.id_hogar = f.id_hogar
    GROUP BY u.departamento, h.tipo_vivienda
)
SELECT departamento, tipo_vivienda, ingreso_promedio, ranking_vivienda
FROM ingreso_departamento
WHERE ranking_vivienda = 1
ORDER BY ingreso_promedio DESC;
]
(Background on this error at: 

### Interpretación
La consulta prioriza la vivienda asociada al mayor ingreso promedio dentro de cada departamento, lo que permite comparar el contexto residencial más favorable. Esta vista ayuda a detectar si ciertos tipos de vivienda coinciden con mejor desempeño laboral según la zona geográfica.

## Consulta 6
¿En qué medida la migración afecta el ingreso laboral y la permanencia en la fuerza de trabajo?

In [ ]:
query_6 = text('''
SELECT
    m.nacio_en_municipio,
    m.razon_migracion,
    e.categoria AS estado_ocupacional,
    CAST(AVG(f.ingreso_laboral) AS NUMERIC(18, 2)) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_migracion m ON m.id_migracion = f.id_migracion
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
GROUP BY m.nacio_en_municipio, m.razon_migracion, e.categoria
ORDER BY ingreso_promedio DESC
LIMIT 20;
''')

with engine.connect() as conn:
    df6 = conn.execute(query_6).fetchall()
    print(df6[:20])


ProgrammingError: (psycopg2.errors.UndefinedColumn) column m.nacion_en_municipio does not exist
LINE 3:     m.nacion_en_municipio,
            ^
HINT:  Perhaps you meant to reference the column "m.nacio_en_municipio".

[SQL: 
SELECT
    m.nacion_en_municipio,
    m.razon_migracion,
    e.categoria AS estado_ocupacional,
    ROUND(AVG(f.ingreso_laboral), 2) AS ingreso_promedio,
    COUNT(*) AS personas
FROM fact_situacion_laboral f
JOIN dim_migracion m ON m.id_migracion = f.id_migracion
JOIN dim_estado_ocupacional e ON e.id_estado = f.id_estado
GROUP BY m.nacion_en_municipio, m.razon_migracion, e.categoria
ORDER BY ingreso_promedio DESC
LIMIT 20;
]
(Background on this error at: https://sqlalche.me/e/21/f405)

### Interpretación
La migración parece relacionarse con diferencias en el ingreso promedio y la categoría ocupacional, lo que sugiere que la trayectoria migratoria puede influir en la participación laboral. Aunque el resultado no prueba causalidad, sí ofrece una señal útil para profundizar en segmentaciones por movilidad y tipo de empleo.